## Imports

In [41]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.cluster import DBSCAN, KMeans

import plotly.express as px
import plotly.graph_objects as go

## Parámetros

In [42]:
# Rutas de archivos
DATA_DIR = Path("data")
BACKGROUND_FILE = "carretera.csv"
POINTCLOUD_FILE = "coche_coche_moto.csv"

# Parámetros de filtrado
BACKGROUND_THRESHOLD = 300

# Parámetros de DBSCAN
DBSCAN_EPS = 1
DBSCAN_MIN_SAMPLES = 10

## Función de visualización

In [43]:
def show_pointcloud(data: pd.DataFrame, title: str = 'Point Cloud', color_col: str = 'z', is_cluster: bool = False):
    """
    Muestra la nube de puntos en 3D integrada en el notebook usando Plotly.
    
    Parámetros:
    - data: DataFrame que contiene al menos las columnas 'x', 'y', 'z'.
    - title: Título del gráfico.
    - color_col: Columna por la que colorear los puntos.
    - is_cluster: True si color_col representa etiquetas de clusters (categorías), 
                  False si es una variable continua (como 'z' o 'range').
    """
    if len(data) == 0:
        print("No hay datos para mostrar.")
        return

    df_plot = data.copy()

    if is_cluster and color_col in df_plot.columns:
        df_plot[color_col] = df_plot[color_col].astype(str)

        fig = px.scatter_3d(df_plot, x='x', y='y', z='z',
                            color=color_col, title=title,
                            color_discrete_sequence=px.colors.qualitative.Plotly)
    else:
        fig = px.scatter_3d(df_plot, x='x', y='y', z='z',
                            color=color_col if color_col in df_plot.columns else None,
                            title=title,
                            color_continuous_scale='Viridis')
        fig.update_layout(coloraxis_showscale=False)

    fig.update_traces(marker=dict(size=2, opacity=0.8),
                      selector=dict(mode='markers'))
    
    fig.update_layout(margin=dict(l=0, r=0, b=0, t=40), 
                      scene=dict(xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False), aspectmode='data'),
                      height=750)

    fig.show()

## Clase PointFilter

In [44]:
class PointFilter:
    def __init__(self, background_df: pd.DataFrame, threshold: float = 300):
        self.background = background_df
        self.threshold = threshold

    def remove_background(self, data: pd.DataFrame) -> pd.DataFrame:
        """
        Elimina el fondo usando carretera.csv como referencia.

        IMPORTANTE:
        Esta función debe ejecutarse antes de limpiar/eliminar filas,
        porque necesita mantener la correspondencia punto a punto
        entre ambos frames.

        Si un punto del frame actual está significativamente más cerca
        que el mismo rayo en carretera.csv, se considera un objeto nuevo.
        """

        if len(data) != len(self.background):
            raise ValueError(
                "El frame actual y carretera.csv deben tener mismo número de puntos."
            )

        current_range = data["range"].to_numpy(dtype=float)
        background_range = self.background["range"].to_numpy(dtype=float)

        difference = background_range - current_range

        mask = (
            np.isfinite(current_range)
            & np.isfinite(background_range)
            & (current_range > 0)
            & (background_range > 0)
            & (difference > self.threshold)
        )

        return data.loc[mask].copy()


    def clean_data(self, data: pd.DataFrame) -> pd.DataFrame:
        """
        Elimina puntos inválidos:
        - NaN
        - infinitos
        - range <= 0
        """

        data = data.copy()

        mask = (
            np.isfinite(data["x"])
            & np.isfinite(data["y"])
            & np.isfinite(data["z"])
        )

        if "range" in data.columns:
            mask &= (
                np.isfinite(data["range"])
                & (data["range"] > 0)
            )

        return data.loc[mask].copy()

    def filter_noise(self, data: pd.DataFrame) -> pd.DataFrame:
        """
        Filtrado adicional después de eliminar el fondo.

        Se eliminan:
        - puntos inválidos
        - puntos demasiado lejanos
        - puntos fuera de una zona razonable de trabajo
        """

        data = self.clean_data(data)

        # Limitar puntos demasiado lejanos
        if "range" in data.columns:
            data = data[data["range"] < 40000]

        # Región de interés amplia
        data = data[
            (data["x"] > -30) & (data["x"] < 30) &
            (data["y"] > -10) & (data["y"] < 15) &
            (data["z"] > -10) & (data["z"] < 10)
        ]

        return data.copy()

    def process(self, data: pd.DataFrame, is_moto_case: bool = False) -> pd.DataFrame:
        """
        Aplica el filtrado completo
        """

        if is_moto_case:
            current = data["range"].to_numpy(dtype=float)
            reference = self.background["range"].to_numpy(dtype=float)
            keep = (
                np.isfinite(current) & np.isfinite(reference) & (current > 0)
                & ((reference == 0) | ((reference > 0) & (reference - current > self.threshold)))
            )
            filtered_data = data.loc[keep].copy()
        else:
            filtered_data = self.remove_background(data)

        return self.filter_noise(filtered_data)

## Clase DetectorDBSCAN

In [45]:
class DetectorDBSCAN:
    def __init__(self, eps: float = 1.0, min_samples: int = 10):
        self.eps = eps
        self.min_samples = min_samples
        self.model = DBSCAN(eps=self.eps, min_samples=self.min_samples)
        
    def fit_predict(self, data: pd.DataFrame) -> pd.DataFrame:
        """
        Aplica DBSCAN, asigna las etiquetas de cluster y elimina los puntos 
        marcados como ruido (etiqueta -1).
        """
        if len(data) == 0:
            return data
            
        coords = data[["x", "y", "z"]].to_numpy()
        labels = self.model.fit_predict(coords)
        
        result = data.copy()
        result["cluster_dbscan"] = labels
        
        ruido = len(result[result["cluster_dbscan"] == -1])
        print(f"DBSCAN -> Puntos analizados: {len(result)} | Puntos de ruido eliminados: {ruido}")
        
        # Nos quedamos solo con los puntos que pertenecen a algún clúster válido
        clean_data = result[result["cluster_dbscan"] != -1].copy()
        return clean_data

## Escena Original

In [46]:
target_path = DATA_DIR / POINTCLOUD_FILE
if not target_path.is_file():
    target_path = Path(POINTCLOUD_FILE)

if not target_path.is_file():
    print(f"Archivo {POINTCLOUD_FILE} no encontrado en {DATA_DIR} ni en el directorio actual.")
else:
    print(f"Archivo cargado: {target_path.name}")

    df_raw = pd.read_csv(target_path)
    print(f"Número de puntos: {len(df_raw)}")

    show_pointcloud(df_raw, title='Pointcloud sin filtrar', color_col='z', is_cluster=False)

Archivo cargado: coche_coche_moto.csv
Número de puntos: 131072


## Escena Filtrada

In [47]:
background_path = DATA_DIR / BACKGROUND_FILE
if not background_path.is_file():
    background_path = Path(BACKGROUND_FILE)

df_background = pd.read_csv(background_path)
point_filter = PointFilter(background_df=df_background, threshold=BACKGROUND_THRESHOLD) 

is_moto_case = (POINTCLOUD_FILE == "coche_coche_moto.csv")
df_filtered = point_filter.process(df_raw, is_moto_case=is_moto_case)

print(f"Es caso de coche_coche_moto: {is_moto_case}")
print(f"Número de puntos después del filtrado: {len(df_filtered)}")

show_pointcloud(df_filtered, title='Pointcloud filtrada', color_col='z', is_cluster=False)

Es caso de coche_coche_moto: True
Número de puntos después del filtrado: 4577


## Clusters DBSCAN

In [48]:
detctor_dbscan = DetectorDBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES)

df_clean = detctor_dbscan.fit_predict(df_filtered)

show_pointcloud(df_clean, title='Pointcloud con DBSCAN', color_col='cluster_dbscan', is_cluster=True)

DBSCAN -> Puntos analizados: 4577 | Puntos de ruido eliminados: 403
